# Amazon Prime Video — OTT Streaming Content Trend Analysis

**Objective:** Explore the Prime Video catalog to identify genre, rating, release-year, regional and duration patterns, then translate those patterns into content-strategy insights.

**Deliverables:** EDA + 8 visualizations + SQL + Excel + Power BI + strategy memo.


In [ ]:
# 1. Import libraries
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)

os.makedirs("../visualizations", exist_ok=True)


In [ ]:
# 2. Load dataset
df = pd.read_csv("../data/amazon_prime_titles.csv")

print("Shape:", df.shape)
display(df.head())


In [ ]:
# 3. Data quality audit
print("Duplicate rows:", df.duplicated().sum())
display(df.isna().sum().sort_values(ascending=False).to_frame("missing_values"))
display(df.describe(include="all").T.head(20))


In [ ]:
# 4. Clean and engineer fields
data = df.copy()

# Standardize strings
for col in ["type", "title", "director", "cast", "country", "rating", "duration", "listed_in"]:
    if col in data.columns:
        data[col] = data[col].astype("string").str.strip()

# Convert release year
data["release_year"] = pd.to_numeric(data["release_year"], errors="coerce")

# Parse date_added where available
data["date_added"] = pd.to_datetime(data["date_added"], errors="coerce")

# Extract numeric duration
data["duration_value"] = pd.to_numeric(
    data["duration"].str.extract(r"(\d+)")[0],
    errors="coerce"
)

data["duration_type"] = np.where(
    data["type"].eq("Movie"), "Minutes",
    np.where(data["type"].eq("TV Show"), "Seasons", "Other")
)

# Primary genre
data["primary_genre"] = (
    data["listed_in"]
    .fillna("Unknown")
    .str.split(", ")
    .str[0]
)

# Release decade
data["release_decade"] = (data["release_year"] // 10) * 10

data.head()


## KPI Summary

In [ ]:
# 5. KPI summary
movies = data[data["type"].eq("Movie")].copy()
shows = data[data["type"].eq("TV Show")].copy()

print("Total titles:", len(data))
print("Movies:", len(movies))
print("TV Shows:", len(shows))
print("Unique genres:", data["primary_genre"].nunique())
print("Average movie duration:", round(movies["duration_value"].mean(), 2), "minutes")
print("Most common rating:", data["rating"].mode(dropna=True).iloc[0] if data["rating"].notna().any() else "N/A")


## Visualization 1 — Movies vs TV Shows

In [ ]:
type_counts = data["type"].value_counts()

plt.figure(figsize=(7,5))
ax = sns.barplot(x=type_counts.index, y=type_counts.values)
ax.set_title("Movies vs TV Shows")
ax.set_xlabel("Content Type")
ax.set_ylabel("Number of Titles")
plt.tight_layout()
plt.savefig("../visualizations/01_content_type.png", dpi=200)
plt.show()


## Visualization 2 — Top Genres

In [ ]:
genre_list = []
for value in data["listed_in"].dropna():
    genre_list.extend([g.strip() for g in str(value).split(",")])

genre_counts = pd.Series(genre_list).value_counts()

plt.figure(figsize=(10,6))
ax = sns.barplot(
    x=genre_counts.head(10).values,
    y=genre_counts.head(10).index
)
ax.set_title("Top 10 Genres")
ax.set_xlabel("Number of Titles")
ax.set_ylabel("Genre")
plt.tight_layout()
plt.savefig("../visualizations/02_top_genres.png", dpi=200)
plt.show()


## Visualization 3 — Release-Year Trend

In [ ]:
yearly = (
    data.dropna(subset=["release_year"])
        .groupby("release_year")
        .size()
        .reset_index(name="titles")
)

plt.figure(figsize=(12,6))
ax = sns.lineplot(data=yearly, x="release_year", y="titles")
ax.set_title("Amazon Prime Content by Release Year")
ax.set_xlabel("Release Year")
ax.set_ylabel("Number of Titles")
plt.tight_layout()
plt.savefig("../visualizations/03_release_year_trend.png", dpi=200)
plt.show()


## Visualization 4 — Top Countries

In [ ]:
country_list = []
for value in data["country"].dropna():
    country_list.extend([c.strip() for c in str(value).split(",")])

country_counts = pd.Series(country_list).value_counts()

plt.figure(figsize=(10,6))
ax = sns.barplot(
    x=country_counts.head(10).values,
    y=country_counts.head(10).index
)
ax.set_title("Top 10 Countries by Content Representation")
ax.set_xlabel("Number of Titles")
ax.set_ylabel("Country")
plt.tight_layout()
plt.savefig("../visualizations/04_top_countries.png", dpi=200)
plt.show()


## Visualization 5 — Rating Distribution

In [ ]:
rating_counts = data["rating"].fillna("Unknown").value_counts().head(15)

plt.figure(figsize=(10,7))
ax = sns.barplot(x=rating_counts.values, y=rating_counts.index)
ax.set_title("Content Rating Distribution")
ax.set_xlabel("Number of Titles")
ax.set_ylabel("Rating")
plt.tight_layout()
plt.savefig("../visualizations/05_rating_distribution.png", dpi=200)
plt.show()


## Visualization 6 — Movie Duration

In [ ]:
movie_duration = movies["duration_value"].dropna()

plt.figure(figsize=(10,6))
ax = sns.histplot(movie_duration, bins=30, kde=True)
ax.set_title("Movie Duration Distribution")
ax.set_xlabel("Duration (minutes)")
ax.set_ylabel("Number of Movies")
plt.tight_layout()
plt.savefig("../visualizations/06_movie_duration.png", dpi=200)
plt.show()

print(movie_duration.describe())


## Visualization 7 — Genre Trend Over Time

In [ ]:
top_genres = genre_counts.head(5).index.tolist()

genre_year = (
    data[data["primary_genre"].isin(top_genres)]
    .dropna(subset=["release_year"])
    .groupby(["release_year", "primary_genre"])
    .size()
    .reset_index(name="titles")
)

plt.figure(figsize=(12,7))
ax = sns.lineplot(
    data=genre_year,
    x="release_year",
    y="titles",
    hue="primary_genre"
)
ax.set_title("Top Genre Trends Over Release Years")
ax.set_xlabel("Release Year")
ax.set_ylabel("Number of Titles")
plt.tight_layout()
plt.savefig("../visualizations/07_genre_trend.png", dpi=200)
plt.show()


## Visualization 8 — Movies vs TV Shows by Year

In [ ]:
type_year = (
    data.dropna(subset=["release_year"])
        .groupby(["release_year", "type"])
        .size()
        .reset_index(name="titles")
)

plt.figure(figsize=(12,7))
ax = sns.lineplot(
    data=type_year,
    x="release_year",
    y="titles",
    hue="type"
)
ax.set_title("Movies vs TV Shows by Release Year")
ax.set_xlabel("Release Year")
ax.set_ylabel("Number of Titles")
plt.tight_layout()
plt.savefig("../visualizations/08_type_by_year.png", dpi=200)
plt.show()


## Bonus Statistical Analysis

This section adds a statistical component to strengthen the internship submission.


In [ ]:
# 9. Correlation: release year vs movie duration
corr_df = movies[["release_year", "duration_value"]].dropna()

if len(corr_df) >= 3:
    corr, p_value = stats.pearsonr(
        corr_df["release_year"],
        corr_df["duration_value"]
    )
    print("Pearson correlation:", round(corr, 4))
    print("p-value:", p_value)
else:
    print("Not enough complete observations for correlation.")


In [ ]:
# 10. Compare movie durations by broad release period
movies_stat = movies.dropna(subset=["release_year", "duration_value"]).copy()
movies_stat["period"] = np.where(
    movies_stat["release_year"] >= 2010,
    "2010+",
    "Before 2010"
)

group_a = movies_stat.loc[movies_stat["period"].eq("2010+"), "duration_value"]
group_b = movies_stat.loc[movies_stat["period"].eq("Before 2010"), "duration_value"]

if len(group_a) > 1 and len(group_b) > 1:
    t_stat, p_value = stats.ttest_ind(group_a, group_b, equal_var=False)
    print("2010+ average:", round(group_a.mean(), 2))
    print("Before 2010 average:", round(group_b.mean(), 2))
    print("Welch t-test statistic:", round(t_stat, 4))
    print("p-value:", p_value)
else:
    print("Not enough observations for the t-test.")


# Content Strategy Memo — Evidence Framework

Use the actual outputs above to fill the three recommendations.

### Insight 1 — Genre portfolio
**Observation:** Identify the top genres and whether their representation changes over time.

**Business implication:** A concentrated genre mix can indicate established catalog strengths while also highlighting areas for portfolio diversification.

**Action:** Maintain strong coverage in established categories while testing selected emerging categories.

### Insight 2 — Regional mix
**Observation:** Identify the leading countries and underrepresented regions.

**Business implication:** Country concentration may indicate where the catalog is already strong and where localized acquisition could expand variety.

**Action:** Investigate regional production partnerships and localized content opportunities.

### Insight 3 — Content freshness / format
**Observation:** Use release-year trends and movie-duration distribution.

**Business implication:** The balance between recent and older content, plus format length, can inform programming and acquisition decisions.

**Action:** Balance newer releases with evergreen library titles and format-specific content experiments.

> Important: Replace generic wording with the exact numbers produced by this notebook before submitting.
